# Lecture 8.5 — Response Sanitization with `after_model_callback`

**Design Pattern:** Request / Response Modification (P5)  
**Callback used:** `after_model_callback`  
**Applied to:** `cost_cutter_agent`

In this lecture we add a response sanitization callback to `cost_cutter_agent`. The LLM reliably produces *almost-correct* JSON — but with two consistent flaws that break downstream parsing:

1. **Markdown code fences** — wraps the JSON in ` ```json ... ``` ` blocks  
2. **String costs** — writes cost values as strings (`"cost": "5000"`) instead of numbers (`"cost": 5000`)

The `after_model_callback` intercepts the raw LLM response before the framework processes it further, strips the fences, coerces all cost values to floats, and returns a clean `LlmResponse`. `accountant_agent`'s `sum_costs` tool then always receives valid, parseable numbers.

---
**Changes from Lecture 8.4 (the complete diff):**
1. New import: `LlmResponse` from `google.adk.models.lite_llm`
2. New function: `sanitize_cost_cutter_response` — fence-stripping + cost coercion
3. One new keyword argument on `cost_cutter_agent`: `after_model_callback=sanitize_cost_cutter_response`
4. One new standalone demo cell — runs the callback directly on mock responses

---
**Expected output — fences present:**
```
[SANITIZE] Stripped markdown fences from cost_cutter_agent response
```
**Expected output — string costs present:**
```
[SANITIZE] Coerced 2 string cost(s) to float in cost_cutter_agent response
```
**Expected output — already clean:**
```
[SANITIZE] Response already clean. Passing through.
```

## ⚙️ 1. Setup: Install Libraries

Pinning the version ensures our code will always work as expected.

In [ ]:
!pip install google-adk==1.29.0 -q

## 🔑 2. Authentication: Configure Your API Key

In [ ]:
import os
from getpass import getpass

api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully!")

## 🤖 3. Model Configuration

Define the model names once here. To upgrade to a newer model in the future,
change these two constants — nothing else in the notebook needs to touch.

In [ ]:
# ── Model Configuration ───────────────────────────────────────────────────────
# Change these two constants to swap models across the entire notebook.
# To upgrade to a newer model in the future, update AGENT_MODEL and JUDGE_MODEL here.

AGENT_MODEL = "gemini-2.5-flash"  # used by all workflow agents
JUDGE_MODEL = "gemini-2.5-flash"   # used by the safety judge


## 🪝 3. [CARRIED OVER from 8.3 & 8.4] Define the Observability Callbacks

These two callbacks are unchanged from Lecture 8.3 and carried forward through 8.4.  
They fire at the **agent** boundary (entry and exit).  
The new `sanitize_cost_cutter_response` in the next cell fires at the **model** boundary — a different, deeper hook that intercepts the raw LLM output before the framework processes it.

All three callbacks coexist and fire simultaneously during a live run.

In [ ]:
from datetime import datetime
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types

# A module-level variable so log_agent_exit can calculate elapsed time.
_workflow_start_time: datetime = None


def log_agent_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    before_agent_callback for spending_proposer_agent.

    Fires once, right before the first LLM call in the entire workflow.
    Records the start time and prints a structured ENTRY log line.

    Returns None — the agent proceeds normally. Nothing is skipped.
    """
    global _workflow_start_time
    _workflow_start_time = datetime.now()  # Capture start time for later

    # --- Read from CallbackContext ---
    agent_name    = callback_context.agent_name      # e.g. 'spending_proposer_agent'
    invocation_id = callback_context.invocation_id   # unique UUID for this run
    state_keys    = list(callback_context.state.to_dict().keys())  # what's in memory so far
    timestamp     = _workflow_start_time.strftime("%H:%M:%S")

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[ENTRY] {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        state_keys : {state_keys}")
    print("="*60)

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — proceed with the agent as normal.'
    return None


def log_agent_exit(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    after_agent_callback for plan_retriever_agent.

    Fires once, right after the last agent in the workflow completes.
    Calculates total elapsed time and prints a structured EXIT log line.

    Returns None — the agent's output is used unchanged. Nothing is replaced.
    """
    now        = datetime.now()
    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    state_keys = list(callback_context.state.to_dict().keys())
    timestamp  = now.strftime("%H:%M:%S")

    # Calculate duration only if log_agent_entry ran first
    if _workflow_start_time is not None:
        elapsed = (now - _workflow_start_time).seconds
        duration_str = f"{elapsed}s"
    else:
        duration_str = "n/a"

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[EXIT]  {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        duration   : {duration_str}")
    print(f"        state_keys : {state_keys}")
    print("="*60 + "\n")

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — use the agent's real output as-is.'
    return None

## 🛡️ 4. [CARRIED OVER from 8.4] Define the LLM-as-Judge Guardrail

This cell is **unchanged from Lecture 8.4**. The guardrail fires at the **agent** boundary on `budget_optimizer_workflow` — before any sub-agent starts.

### Why it is still here

The 8.5 sanitizer (`after_model_callback`) operates at the **model** boundary — a completely different level. Both callbacks coexist without interference:

| Callback | Hook | Fires on | Boundary |
|---|---|---|---|
| `guardrail_before_workflow` | `before_agent_callback` | `budget_optimizer_workflow` | Agent |
| `sanitize_cost_cutter_response` | `after_model_callback` | `cost_cutter_agent` | Model |

### Return value contract (reminder)
- Return `None` → topic is safe, workflow runs normally  
- Return `Content` → entire workflow cancelled instantly — no sub-agent ever starts


In [ ]:
# ============================================================
# Lecture 8.4 — LLM-as-Judge Guardrail  [CARRIED OVER — UNCHANGED]
# Design Patterns: Guardrails & Policy Enforcement (P1)
#                  Conditional Skipping of Steps (P6)
# ============================================================

from google.genai import client as genai_client

# Initialise a direct Gemini client for the safety judge.
# This is a raw API call — completely separate from the ADK runner.
safety_client = genai_client.Client()

# ── Judge prompts ─────────────────────────────────────────────────────────
# Two-stage design:
#   Prompt 1 — binary verdict (YES/NO). Fast, cheap, used on every request.
#   Prompt 2 — human-readable explanation. Only called when verdict is YES,
#              so clean topics pay no extra cost.

SAFETY_VERDICT_PROMPT = """
You are a strict safety officer for an event planning company.
Evaluate the following event planning request.

Does it involve any of the following:
- Dangerous or high-risk activities
- Weapons, arms, or military equipment
- Illegal substances or narcotics
- Illegal activities of any kind
- Activities that cannot be commercially insured
- Adult-only or explicit content
- Anything that exposes the company to legal or reputational risk

Reply with EXACTLY one word — either YES or NO.
No explanation. No punctuation. Just the single word.

Event request: "{topic}"
"""

SAFETY_REASON_PROMPT = """
You are a polite but firm safety officer for an event planning company.
A client has requested help planning an event, but it has been flagged as
unsafe or inappropriate for our business.

Write a short, professional refusal message (2-3 sentences) addressed to
the client. Explain specifically why this type of event falls outside what
the company can assist with. Be clear but courteous. Do not offer
workarounds or alternatives.

Event request: "{topic}"
"""


def guardrail_before_workflow(
    callback_context: CallbackContext,
) -> Optional[types.Content]:
    """
    before_agent_callback on budget_optimizer_workflow (the SequentialAgent).

    Two-stage LLM-as-judge:
      Stage 1 — fast binary verdict (YES/NO) on every request.
      Stage 2 — rich refusal explanation, only when Stage 1 says YES.

    The explanation is written into state["refusal_reason"] so the runner
    can surface it as the final response instead of "No plan found."

    Placed on the SequentialAgent so it fires once before ANY sub-agent
    starts. Returning Content cancels the entire workflow instantly.
    """
    topic = callback_context.state.get("topic", "")

    print("\n" + "─" * 60)
    print(f"[SAFETY JUDGE] Evaluating topic: '{topic}'")

    # ── Stage 1: Binary verdict ───────────────────────────────────────────
    verdict_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_VERDICT_PROMPT.format(topic=topic),
    )
    verdict = verdict_response.text.strip().upper()
    print(f"[SAFETY JUDGE] Verdict         : {verdict}")

    if "YES" not in verdict:
        print(f"  └─ ✅ SAFE — starting workflow.")
        print("─" * 60)
        return None                        # topic is safe, proceed normally

    # ── Stage 2: Rich explanation (only reached when blocked) ─────────────
    print(f"[SAFETY JUDGE] Generating refusal explanation...")
    reason_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_REASON_PROMPT.format(topic=topic),
    )
    refusal_reason = reason_response.text.strip()
    print(f"[SAFETY JUDGE] Reason          : {refusal_reason}")
    print(f"  └─ 🚫 BLOCKED — high-risk topic. Workflow cancelled.")
    print("─" * 60)

    # Write the rich explanation into session state.
    # The runner reads state["refusal_reason"] when state["final_presentation"]
    # is absent — this is how the final response reaches the user.
    callback_context.state["refusal_reason"] = refusal_reason

    # Return Content to cancel the entire workflow.
    return types.Content(
        role="model",
        parts=[types.Part(text=refusal_reason)],
    )


## 🧹 5. [NEW] Define the Response Sanitization Callback

This is the **only new code in this lecture**.

### Why `after_model_callback`?

The `after_model_callback` fires after the LLM has produced its response but **before** the ADK framework processes it further. This makes it the ideal hook for cleaning up formatting quirks that the LLM introduces reliably.

### Why `cost_cutter_agent` specifically?

`cost_cutter_agent` runs inside the `LoopAgent` and writes to `output_key="current_plan"`. That value is read directly by `accountant_agent`'s `sum_costs` tool on the next iteration. One bad string cost causes `sum_costs` to silently return a wrong total or throw a type error — corrupting the entire refinement loop.

### Two flaws, one callback

| Flaw | Example (raw) | Example (fixed) |
|---|---|---|
| Markdown fences | `` ```json\n{...}\n``` `` | `{...}` |
| String costs | `"cost": "8000"` | `"cost": 8000.0` |

### Return value contract

| Condition | Return value |
|---|---|
| Function call response | `None` — pass through, never touch tool calls |
| Response already clean | `None` — original passes through unchanged |
| Fences or string costs found | New `LlmResponse` with cleaned content |

### The immutability rule

Always build a **new** `LlmResponse` — never mutate the original in-place. Other callbacks in the chain may hold references to the original object. Mutating it silently corrupts their view.

### Fires on every loop iteration

The callback fires every time `cost_cutter_agent`'s LLM is called — which means it fires on every loop iteration. Watch for the `[SANITIZE]` log line repeating across iterations in the live run output.

In [ ]:
# ============================================================
# Lecture 8.5 — Response Sanitization: after_model_callback
# Design Pattern: Request / Response Modification (P5)
# ============================================================



## 🛠️ 6. Define Workflow Tools

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
import json
from google.adk.tools import ToolContext

def sum_costs(costs: list[float]) -> float:
    """Calculates the sum of a list of numbers."""
    print(f"  [Tool Call] sum_costs on the list: {costs}")
    return sum(costs)

def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the plan is approved and within budget."""
    print(f"  [Tool Call] Budget approved. Terminating loop: {json.dumps(tool_context.state.to_dict())}")
    tool_context.actions.escalate = True
    return None

## 7. Create Tool Wrappers

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_agent",
    model=AGENT_MODEL,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)

google_search_tool = AgentTool(agent=google_search_agent)

## 📝 8. Create Agents

**One change from Lecture 8.4** — highlighted with `# <- 8.5 CHANGE`:

- `cost_cutter_agent` — gains `after_model_callback=sanitize_cost_cutter_response`.
- `spending_proposer_agent`, `accountant_agent`, `plan_retriever_agent` — all unchanged from 8.4.

Three callbacks now fire across the workflow simultaneously:
- `guardrail_before_workflow` — on `budget_optimizer_workflow` (before_agent_callback)
- `log_agent_entry` — on `spending_proposer_agent` (before_agent_callback)
- `sanitize_cost_cutter_response` — on `cost_cutter_agent` (after_model_callback)  ← **8.5 CHANGE**
- `log_agent_exit` — on `plan_retriever_agent` (after_agent_callback)

In [ ]:
COMPLETION_PHRASE = "The plan is within the budget."

# Agent 1: Proposes the initial, expensive plan (runs once).
# Guardrail has moved to the SequentialAgent — this agent is now clean.
# before_agent_callback=log_agent_entry carried over from 8.3 unchanged.
spending_proposer_agent = Agent(
    name="spending_proposer_agent",
    model=AGENT_MODEL,
    tools=[google_search],
    instruction="""
    You are a luxury event planner. For a {{topic}}, find a high-end venue and a gourmet catering service.

    Output a JSON object with items and their estimated costs, like:
    {"venue": {"name": "The Ritz London", "cost": 10000}, "catering": {"name": "Gourmet Chefs Inc.", "cost": 5000}}
    """,
    output_key="current_plan",
    before_agent_callback=log_agent_entry,   # ← 8.3/8.4 (unchanged)
)

# Agent 2 (in loop): The "Accountant" that critiques the plan.
# Unchanged from Section 5.
accountant_agent = Agent(
    name="accountant_agent",
    model=AGENT_MODEL,
    tools=[sum_costs],
    instruction=f"""
    You are a meticulous accountant. Your budget is {{{{budget}}}}.
    The current plan is: {{{{current_plan}}}}

    Extract the costs from the plan and use the `sum_costs` tool to get the total.
    - IF the total cost is > {{{{budget}}}}, output a critique like: "This plan is over budget by [amount]. Find a cheaper [item]."
    - ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key="critique",
)

# Agent 3 (in loop): The "Cost Cutter" that refines the plan.
# after_model_callback added in 8.5 — sanitizes JSON output before accountant reads it.
cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=AGENT_MODEL,
    tools=[google_search_tool, exit_loop],
    instruction=f"""
    You are a cost-cutting expert. You must refine a plan based on a critique.
    The critique is: {{{{critique}}}}
    The current plan is: {{{{current_plan}}}}

    - IF the critique is '{COMPLETION_PHRASE}'
        1. You MUST call the `exit_loop` tool with no arguments.
        2. After calling exit_loop, output the current plan EXACTLY as-is, character for character,
           with no modifications, no acknowledgements, no commentary, and no extra text. Do not summarize it.
           Do not rephrase it. Do not add "Budget approved" or any other text.
           Just echo {{{{current_plan}}}} verbatim.
    - ELSE, read the critique to identify the overpriced item. Use your search tool to find a cheaper alternative for that item.
      Output a new JSON object with the updated plan.
    """,
    output_key="current_plan",
    after_model_callback=sanitize_cost_cutter_response,   # <- 8.5 CHANGE
)

# Agent 4: Presents the final approved plan (runs once after loop).
# Unchanged from 8.4.
plan_retriever_agent = Agent(
    name="plan_retriever_agent",
    model=AGENT_MODEL,
    instruction="""
    You are a plan finalizer. Your only job is to present the final, approved plan.
    The plan is available in the context variable `{{current_plan}}`.

    Your output must be the content of the final plan presented in a clear and easy-to-read format.
    """,
    tools=[],
    output_key="final_presentation",
    after_agent_callback=log_agent_exit,   # ← 8.3/8.4 (unchanged)
)


## 🔄 9. Assemble the Loop and Sequential Workflows

Unchanged from Lecture 8.4. The new callback lives on `cost_cutter_agent` inside the loop — no changes needed here.

In [ ]:
from google.adk.agents import SequentialAgent, LoopAgent

# Unchanged from Section 5.
budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3,
)

# ← 8.4 (unchanged): guardrail still lives on the SequentialAgent.
# 8.5 change is on cost_cutter_agent, not here.
budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[spending_proposer_agent, budget_refinement_loop, plan_retriever_agent],
    before_agent_callback=guardrail_before_workflow,   # ← 8.4 (unchanged)
)


## 🚀 10. Build the Execution Engine

Unchanged from Lecture 8.4. The sanitization callback fires automatically inside the loop — the runner does not need to know about it.

In [ ]:
from IPython.display import display, Markdown

from google.adk.sessions import Session
from google.genai.types import Content, Part
from google.adk.runners import Runner

async def run_agent_query(agent: Agent, query: str, topic: str, budget: str, session: Session, user_id: str):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
            state_delta={"budget": budget, "topic": topic, "COMPLETION_PHRASE": COMPLETION_PHRASE}
        ):
            pass
    except Exception as e:
        final_response = f"An error occurred: {e}"
        return final_response

    # Read the final response from session state.
    #
    # Two possible paths through the workflow:
    #
    #   ✅ CLEAN topic  → plan_retriever_agent runs and writes final_presentation.
    #                     We read that.
    #
    #   🚫 BLOCKED topic → guardrail cancels the workflow and writes refusal_reason
    #                      into state. plan_retriever never runs, so
    #                      final_presentation is never written. We read
    #                      refusal_reason instead.
    #
    final_session = await session_service.get_session(
        app_name=agent.name,
        user_id=user_id,
        session_id=session.id
    )
    state = final_session.state

    if "final_presentation" in state:
        final_response = state["final_presentation"]
    elif "refusal_reason" in state:
        final_response = state["refusal_reason"]
    else:
        final_response = "No response was generated."

    print("\n" + "-"*50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-"*50 + "\n")

    return final_response


## ✨ 11. Initialize Session Service

Unchanged from Lecture 8.4.

In [ ]:
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
user_id = "adk_event_planner_001"

## 🔬 12. Standalone Callback Demo

Before running the full workflow, let's verify the callback works correctly by calling it directly with mock inputs.

This cell creates three mock `LlmResponse` objects — one with fences, one with string costs, one already clean — and calls `sanitize_cost_cutter_response` on each.  
No runner, no session, no agents required.

**Scenario 1** — Markdown fences present  
**Scenario 2** — String costs present  
**Scenario 3** — Already clean (callback returns `None`)

In [ ]:
# ============================================================
# Standalone demo — call the callback directly with mock data
# ============================================================
from unittest.mock import MagicMock
from google.adk.models import LlmResponse
from google.genai import types as genai_types

def make_mock_response(text):
    """Build a minimal mock LlmResponse containing a single text part."""
    content = genai_types.Content(
        role="model",
        parts=[genai_types.Part(text=text)],
    )
    return LlmResponse(content=content)

def make_mock_context(agent_name):
    """Build a minimal mock CallbackContext with an agent_name."""
    ctx = MagicMock()
    ctx.agent_name = agent_name
    return ctx

ctx = make_mock_context("cost_cutter_agent")

print("=" * 60)
print("SCENARIO 1 — Markdown fences present")
print("=" * 60)
raw_with_fences = (
    "```json\n"
    '{"venue": {"name": "The Standard", "cost": 8000},'
    ' "catering": {"name": "Fresh Co", "cost": 4500}}\n'
    "```"
)
resp1 = make_mock_response(raw_with_fences)
result1 = sanitize_cost_cutter_response(ctx, resp1)
print(f"Input : {repr(raw_with_fences[:60])}...")
print(f"Output: {result1.content.parts[0].text if result1 else None}")

print()
print("=" * 60)
print("SCENARIO 2 — String costs present")
print("=" * 60)
raw_string_costs = (
    '{"venue": {"name": "Ritz", "cost": "10000"},'
    ' "catering": {"name": "Saveur", "cost": "4200"}}'
)
resp2 = make_mock_response(raw_string_costs)
result2 = sanitize_cost_cutter_response(ctx, resp2)
print(f"Input : {raw_string_costs}")
print(f"Output: {result2.content.parts[0].text if result2 else None}")

print()
print("=" * 60)
print("SCENARIO 3 — Already clean (callback returns None)")
print("=" * 60)
raw_clean = (
    '{"venue": {"name": "Ace Hotel", "cost": 7000},'
    ' "catering": {"name": "Plated", "cost": 3500}}'
)
resp3 = make_mock_response(raw_clean)
result3 = sanitize_cost_cutter_response(ctx, resp3)
print(f"Input : {raw_clean}")
print(f"Return value: {result3}  (None means original passes through unchanged)")

## ▶️ 13a. Run — CLEAN Topic (Full Workflow with Sanitization)

The topic `"50 person AI event in New York"` passes the guardrail and runs the full workflow.  
Watch for `[SANITIZE]` log lines appearing **on every loop iteration** as `cost_cutter_agent` produces its output and the callback fires.

You will see all callback layers active simultaneously:
- `[SAFETY JUDGE]` — guardrail evaluates the topic
- `[ENTRY]` — `log_agent_entry` fires as `spending_proposer_agent` starts
- `[SANITIZE]` — `sanitize_cost_cutter_response` fires after each `cost_cutter_agent` LLM call
- `[EXIT]` — `log_agent_exit` fires after `plan_retriever_agent` completes

In [ ]:
async def run_clean_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "50 person AI event in New York"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_clean_topic()

## 🚫 13b. Run — BLOCKED Topic (Guardrail Still Intercepts)

The topic `"weapons convention"` is still blocked by the 8.4 guardrail.  
Notice: `[SANITIZE]` never appears — because `cost_cutter_agent` never runs when the workflow is cancelled at the outermost boundary.

This confirms the sanitization callback only fires when `cost_cutter_agent` actually executes.

In [ ]:
async def run_blocked_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "weapons convention"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_blocked_topic()